# Silver Preparation
# Goal:

# clean and standardize the bronze orders
filter invalid records
derive useful fields
save the silver table

In [0]:
import yaml
from pyspark.sql import functions as F

In [0]:
config_path = "/Workspace/Repos/adb-emily@startsteps.org/redcare-pharmacy--project/config/project_config.yml"

In [0]:

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

In [0]:
catalog_name = config["catalog"]
schema_name = config["schema"]

bronze_table = f"{catalog_name}.{schema_name}.{config['tables']['bronze_orders']}"
silver_table = f"{catalog_name}.{schema_name}.{config['tables']['silver_orders']}"
high_value_threshold = config["rules"]["high_value_threshold"]
     

In [0]:
bronze_df = spark.table(bronze_table)

In [0]:
silver_df = (
    bronze_df
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_amount",
        "order_date"
    )
     .withColumn("order_amount", F.col("order_amount").cast("double"))
     .withColumn("order_date", F.to_date("order_date"))
     .filter(F.col("order_id").isNotNull())
     .filter(F.col("customer_id").isNotNull())
     .filter(F.col("order_amount") > 0)
     .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
     .withColumn("order_day", F.col("order_date"))
     .withColumn(
        "is_high_value_order",
        F.when(F.col("order_amount") >= high_value_threshold, 1).otherwise(0)
    )
)

In [0]:
display(silver_df)

In [0]:
silver_df.write.mode("overwrite").saveAsTable(silver_table)
     